# GND ID Extractor (MARCXML Filter)
This tool allows you to extract specific records from a large GND MARCXML file based on a list of GND IDs provided in an Excel file.

### Why this version?
- **Memory Efficient:** Uses `iterparse` to stream the XML, meaning it can process 5GB+ files on a standard laptop.
- **Fast Lookup:** Uses a Python `set` for O(1) matching speed.

In [ ]:
import pandas as pd
import lxml.etree as ET
import os

# === CONFIGURATION ===
# Update these paths to match your local setup
excel_path = "../data/target_ids.xlsx"
gnd_xml_path = "../data/gnd_authority_file.xml"
output_excel_path = "../outputs/extracted_records.xlsx"

# === 1. LOAD TARGET IDs ===
print(f"✅ Loading GND IDs from '{excel_path}'...")
df_ids = pd.read_excel(excel_path)

# Ensure the ID column is treated as a string set for fast matching
# Note: Change 'GND-ID' to match the column header in your Excel file
target_ids = set(df_ids['GND-ID'].dropna().astype(str).unique())
print(f"✅ Loaded {len(target_ids)} unique IDs to search for.")

# === 2. STREAM PROCESS XML ===
extracted_records_data = []
records_processed = 0

print(f"✅ Starting optimized processing of {gnd_xml_path}...")

try:
    # Using iterparse to avoid loading the whole file into memory
    context = ET.iterparse(gnd_xml_path, events=('end',), tag='{http://www.loc.gov/MARC21/slim}record')
    
    for event, element in context:
        records_processed += 1
        
        # Extract GND ID from Control Field 001
        gnd_id = None
        id_field = element.find(".//{http://www.loc.gov/MARC21/slim}controlfield[@tag='001']")
        if id_field is not None:
            gnd_id = id_field.text.strip()

        # Check if this record is in our target list
        if gnd_id and gnd_id in target_ids:
            project_name = ""
            # Extract Name from Field 110 or 111 (Corporate/Meeting names)
            name_field = element.find(".//{http://www.loc.gov/MARC21/slim}datafield[@tag='110']") or \
                         element.find(".//{http://www.loc.gov/MARC21/slim}datafield[@tag='111']")
            
            if name_field is not None:
                subfields = [sub.text for sub in name_field.findall("{http://www.loc.gov/MARC21/slim}subfield") if sub.text]
                project_name = " ".join(subfields)

            extracted_records_data.append({
                'GND-ID': gnd_id,
                'Project Name': project_name
            })
            
            print(f"🔎 Match found! GND ID: {gnd_id} | Total: {len(extracted_records_data)}")

        # CRITICAL: Clear the element and its ancestors to free up RAM
        element.clear()
        while element.getprevious() is not None:
            del element.getparent()[0]

    # === 3. SAVE RESULTS ===
    print(f"\n✅ Processing finished. Total records scanned: {records_processed}")
    
    output_df = pd.DataFrame(extracted_records_data)
    output_df.to_excel(output_excel_path, index=False)
    
    print(f"✅ Successfully exported {len(output_df)} matches to: {output_excel_path}")

except FileNotFoundError:
    print(f"❌ Error: The file was not found. Please check your file paths.")
except Exception as e:
    print(f"❌ An error occurred: {e}")